# Development: why does the policy struggle to finish?

Preserves the separate full flywheel experiment. Compare checkpoints on the **same development starts**, test executing 2/4/8 actions before replanning, and inspect near-success behavior. Start with a small verification run, then train 50-episode and all-data references. No final-test seeds are used.

Drive persistence is enabled by default. Use a GPU for learning runs. Existing trusted checkpoints can be evaluated without retraining. This notebook benchmarks our custom policy; it does not yet include a verified upstream pretrained policy comparison.


In [ ]:
from pathlib import Path
import subprocess, sys
REPOSITORY = "https://github.com/0-5-blood-prince/robotics-data-flywheel.git"
REVISION = "main"  # Set to a commit SHA to reproduce a particular code version.
ROOT = Path('/content/robotics-data-flywheel')
if any(name.startswith('robot_flywheel') for name in sys.modules):
    raise RuntimeError('Restart the runtime before refreshing project code. Saved Drive checkpoints remain available.')
if not ROOT.exists():
    subprocess.run(['git', 'clone', REPOSITORY, str(ROOT)], check=True)
else:
    if not (ROOT / '.git').exists():
        raise RuntimeError('This directory came from an old ZIP. Start a fresh Colab runtime once to switch to GitHub setup.')
    changes = subprocess.check_output(['git','-C',str(ROOT),'status','--porcelain','--untracked-files=no'],text=True)
    if changes.strip(): raise RuntimeError('Local code changes found; preserve them before refreshing.')
subprocess.run(['git','-C',str(ROOT),'fetch','origin',REVISION],check=True)
subprocess.run(['git','-C',str(ROOT),'checkout','--detach','FETCH_HEAD'],check=True)
CODE_COMMIT = subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip()
print('Code commit:', CODE_COMMIT)
import os, importlib
os.chdir(ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{ROOT}[simulation]'], check=True)
# Activate src in this already-running kernel: editable-install .pth files
# are normally processed only when a Python interpreter starts.
sys.path.insert(0, str(ROOT / 'src'))
importlib.invalidate_caches()
import robot_flywheel
assert Path(robot_flywheel.__file__).resolve().is_relative_to(ROOT.resolve())
print('Package ready:', robot_flywheel.__file__)

# Check the native simulator API before dataset preparation.
subprocess.run([sys.executable, '-c',
    "import gymnasium as gym, gym_pusht; e=gym.make('gym_pusht/PushT-v0'); e.reset(seed=0); e.step(e.action_space.sample()); e.close(); print('Simulator ready')"], check=True)


## Persistence and runtime
Set `USE_DRIVE=True` to preserve checkpoints across runtime replacement. Checkpoints are saved every 1,000 updates; rerunning training resumes automatically. Only load checkpoints produced by this project or another trusted source.

In [ ]:
import os, sys, json, torch, numpy as np
os.environ['SDL_VIDEODRIVER'] = 'dummy'
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = Path('/content/drive/MyDrive/robotics-data-flywheel-runs')
else:
    WORK = ROOT / 'runs'
WORK.mkdir(parents=True, exist_ok=True)
(WORK / 'code_commit.txt').write_text(CODE_COMMIT)
print({'python':sys.version,'torch':torch.__version__,'cuda':torch.cuda.is_available(),
       'device':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})
if not torch.cuda.is_available():
    print('Switch to a GPU runtime before substantial training. CPU is suitable for tests only.')
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


## 1. Download and inspect the full-state expert dataset
The SHA-256 is recorded for provenance. Once verified, reuse it as `expected_sha256` on future downloads. It is not an independently authenticated upstream checksum.

In [ ]:
from robot_flywheel.prepare import prepare
from robot_flywheel.data import load_zarr
EPISODES, SOURCE = prepare(ROOT / 'data/pusht')
display(SOURCE)


In [ ]:
import matplotlib.pyplot as plt
example = EPISODES[0]
fig, ax = plt.subplots(1,2,figsize=(11,4))
ax[0].plot(example.states[:,0],example.states[:,1],label='Agent')
ax[0].plot(example.states[:,2],example.states[:,3],label='Object')
ax[0].set(xlim=(0,512),ylim=(0,512),title='Recorded demonstration',aspect='equal'); ax[0].legend()
ax[1].plot(example.actions); ax[1].set(title='Expert target positions',xlabel='Control step',ylabel='Position')
plt.show()


## 2. Compatibility gate
Replay three expert episodes. The legacy reset may shift object pose; the adapter reports both the raw and corrected reset. Missing velocities can still cause divergence. If this check fails, inspect the traces and fix compatibility before training; do not simply increase tolerances to get past the check.

In [ ]:
from robot_flywheel.rollout import check_replay
COMPATIBILITY = check_replay(EPISODES, WORK / 'compatibility', count=3)
display(COMPATIBILITY)
fig, axs = plt.subplots(1,3,figsize=(15,4))
for ax, episode in zip(axs, EPISODES[:3]):
    d = np.load(WORK / 'compatibility' / f'replay_{episode.id}.npz')
    ax.plot(d['recorded'][:,2],d['recorded'][:,3],label='Recorded object')
    ax.plot(d['simulated'][:,2],d['simulated'][:,3],label='Replayed object')
    ax.set(title=f'Episode {episode.id}',aspect='equal'); ax.legend()
plt.show()
assert COMPATIBILITY['passed'], 'Compatibility failed. Inspect report/traces before training.'


## Run plan
`quick` validates this notebook with two updates and two short rollouts. Switch to `development` for useful training. Default development grid: two dataset sizes at 5k updates, three execution prefixes, and five fixed starts = 30 comparison rollouts, plus 10 early screening rollouts. A weak screen stops larger training budgets for that dataset size. DDIM uses 10 model calls per plan instead of 100; policy behavior can change. This is exploratory and uses one training seed.

Training resumes within the same directory. Checkpoint configuration must match. Keep the printed code commit for reproducibility.

In [ ]:
from robot_flywheel.baseline import BaselineConfig, train_baseline
from robot_flywheel.development import compare_checkpoints, plot_traces, side_by_side
MODE = 'quick'  # change to 'development' after verification
assert MODE in ('quick','development')
DEV = WORK / f'development_{MODE}_v2'
DEV.mkdir(parents=True,exist_ok=True)
SIZES = [4] if MODE=='quick' else [50,len(EPISODES)]
UPDATES = [2] if MODE=='quick' else [5000]  # Extend to [5000,20000] only after inspecting the first screen.
SEEDS = [1000,1001] if MODE=='quick' else list(range(1000,1005))
PREFIXES = [2,4,8]
MAX_STEPS = 8 if MODE=='quick' else 300
# Optional existing trusted checkpoints. If populated, training below is skipped.
EXISTING_CHECKPOINTS = {}  # {'baseline_20k': '/path/to/step_0020000.pt'}

INFERENCE_STEPS = 4 if MODE=='quick' else 10  # DDIM screening; confirm with DDPM below.


## Train once per dataset size; retain intermediate checkpoints
The 50-episode and all-data runs share seed, architecture, and optimization settings. Training on all data estimates achievable performance for this implementation; it is not a guaranteed upper bound or an independent architecture reference.

In [ ]:
CHECKPOINTS = dict(EXISTING_CHECKPOINTS)
if not CHECKPOINTS:
    for size in SIZES:
        config = BaselineConfig(initial_episodes=size, updates=UPDATES[0],
            checkpoint_every=2 if MODE=='quick' else 1000,
            batch_size=2 if MODE=='quick' else 128,
            width=16 if MODE=='quick' else 64,
            diffusion_steps=4 if MODE=='quick' else 100)
        run = DEV / f'episodes_{size}'
        for step in UPDATES:
            config.updates=step
            checkpoint=run/f'step_{step:07d}.pt'
            if not checkpoint.exists():
                train_baseline(EPISODES,SOURCE,config,run,compatibility=COMPATIBILITY)
            label=f'{size}eps_{step}updates'
            CHECKPOINTS[label]=str(checkpoint)
            screen=compare_checkpoints({label:str(checkpoint)},DEV/'screening',seeds=SEEDS,
                prefixes=(8,),max_steps=MAX_STEPS,video_count=0,inference_steps=INFERENCE_STEPS)
            display(screen)
            if MODE!='quick' and screen[0]['success_rate']==0 and screen[0]['mean_max_coverage']<.6:
                print('Weak development screen: stop extending this run. Inspect the checkpoint before larger budgets.')
                break
display(CHECKPOINTS)


## Paired checkpoint and execution-prefix evaluation
All combinations use identical starts and inference seed rules. Their trajectories diverge naturally. Each checkpoint is evaluated with 2, 4, and 8 actions executed between replans. Shorter prefixes cost more inference time. Results are cached and resume after interruption.

In [ ]:
RESULTS = compare_checkpoints(CHECKPOINTS, DEV/'evaluation', seeds=SEEDS,
    prefixes=PREFIXES, max_steps=MAX_STEPS, video_count=1,inference_steps=INFERENCE_STEPS)
display(RESULTS)
labels = list(CHECKPOINTS)
fig, axes = plt.subplots(1,2,figsize=(12,4))
for prefix in PREFIXES:
    rows = [next(r for r in RESULTS if r['label']==label and r['execute']==prefix) for label in labels]
    axes[0].plot(labels,[r['success_rate'] for r in rows],marker='o',label=f'execute {prefix}')
    axes[1].plot(labels,[r['mean_max_coverage'] for r in rows],marker='o',label=f'execute {prefix}')
for ax in axes: ax.tick_params(axis='x',rotation=35); ax.set_ylim(0,1); ax.legend()
axes[0].set_ylabel('Success rate');axes[1].set_ylabel('Mean maximum coverage')
fig.tight_layout();plt.show()


## Inspect finishing behavior
Choose a checkpoint and seed. A high maximum but low final coverage suggests losing progress. Frequent action-boundary hits or large changes deserve inspection; these are diagnostics, not automatic failure classifications. Near-success failures reached 90% coverage but never completed.

In [ ]:
LABEL = list(CHECKPOINTS)[-1]
TRACE_SEED = SEEDS[0]
PANELS = [r for r in RESULTS if r['label']==LABEL]
plot_traces(PANELS, seed=TRACE_SEED); plt.show()
from IPython.display import Video, FileLink
VIDEO = side_by_side(PANELS, DEV/'prefix_comparison.mp4',seed=SEEDS[0])
display(Video(str(VIDEO),embed=True,width=1100))
display(FileLink(str(DEV/'evaluation/development_results.csv')))


## Decision before resuming the flywheel
- If later checkpoints improve on the same starts, adjust convergence budgets.
- If shorter prefixes help, freeze a new execution horizon for all arms.
- If all-data training still performs poorly, investigate the model and compare with a verified upstream implementation before launching the full study.
- If coverage approaches the threshold but success remains low, inspect final positioning rather than concluding the selector failed.

These are development results. Any chosen configuration must be frozen before a new held-out final evaluation. The existing full-study notebook and code are preserved.

## Confirm a promising screen before selecting settings
DDIM screening can change performance. Set `RUN_CONFIRMATION=True` to compare the selected checkpoint with the original DDPM sampler on 20 development starts. This is an explicit additional cost, not part of quick validation.

In [ ]:
RUN_CONFIRMATION = False
if RUN_CONFIRMATION:
    chosen = max(RESULTS, key=lambda r: (r['success_rate'],r['mean_max_coverage']))
    CONFIRMATION = compare_checkpoints({chosen['label']:chosen['checkpoint']},DEV/'confirmation',
        seeds=range(1000,1020),prefixes=(chosen['execute'],),max_steps=300,
        video_count=1,inference_steps=None)
    display(CONFIRMATION)
